In [1]:
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from pathlib import Path
from gen_variable_standard_static import \
    metrics_search_for_two_fragments_df
from time import sleep
import pvlib
import pytz
from datetime import date, datetime, timedelta
from datetime import time as py_time

In [2]:
pd.__version__

'2.3.3'

In [3]:
systems_cleaned = pd.read_csv('../../../data/core/systems_cleaned.csv')
systems_cleaned.columns

Index(['system_id', 'system_public_name', 'site_location',
       'timezone_or_utc_offset', 'latitude', 'longitude', 'elevation_m',
       'dc_capacity_kW', 'kg_climate', 'pvcz_composite', 'pvcz_t_rack',
       'pvcz_t_roof', 'pvcz_humidity', 'pvcz_wind', 'tracking', 'type',
       'azimuth', 'tilt', 'first_timestamp', 'last_timestamp', 'years',
       'number_records', 'dataset_size_mb', 'available_sensor_channels',
       'qa_status', 'qa_issue', 'first_year', 'is_prize_data',
       'is_lake_parquet_data', 'is_lake_csv_data', 'has_irradiance_data',
       'has_ambient_temperature_data', 'has_temperature_data',
       'has_power_data', 'has_current_data', 'has_voltage_data', 'has_ac_data',
       'has_dc_data', 'module_type', 'simplified_type', 'system_source',
       'num_days_actual_records', 'sample_year'],
      dtype='object')

In [4]:
metrics_dir = Path("../../../data/raw/parquet-metrics/")
metrics_pq = pq.ParquetDataset(metrics_dir)
metrics_df = metrics_pq.read().to_pandas()
metrics_id_set = set(metrics_df.system_id)

In [5]:
ac_power_metrics = metrics_search_for_two_fragments_df(metrics_df, 'ac', 'pow', 'and')
ac_power_systems = set(ac_power_metrics['system_id'])
two_years_days = 2 * 365
enough_data_systems = systems_cleaned[systems_cleaned['num_days_actual_records'] >= two_years_days]
enough_data_ids = set(enough_data_systems.system_id)
enough_data_parquet_power_systems = enough_data_ids.intersection(ac_power_systems)
enough_data_parquet_power_list = list(enough_data_parquet_power_systems)
enough_data_parquet_power_list.sort()

In [6]:
systems_cleaned_enough_ac_data = systems_cleaned[systems_cleaned['system_id'].isin(enough_data_parquet_power_systems)]

In [7]:
good_time_systems_copied = [4, 10, 33, 36, 50, 51, 1199, 1204, 1283, 1284, 1289, 1310, 1332, 4901, 4902, 4903]

In [9]:
brief_copy = systems_cleaned[systems_cleaned['system_id'].isin(good_time_systems_copied)]
brief_copy[['system_id', 'timezone_or_utc_offset', 'tracking', 'tilt', 'azimuth']]

,system_id,timezone_or_utc_offset,tracking,tilt,azimuth
2,4,7,fixed,40.00,180.0
3,10,7,fixed,40.00,180.0
4,33,7,fixed,40.00,180.0
7,36,7,fixed,45.00,180.0
8,50,7,fixed,45.00,158.0
9,51,7,fixed,45.00,158.0
10,1199,America/New_York,fixed,20.00,180.0
15,1204,America/New_York,fixed,5.00,140.0
68,1283,7,fixed,10.00,165.0
69,1284,7,fixed,40.00,180.0


In [10]:
systems_cleaned[systems_cleaned['system_id'] == 4]

,system_id,system_public_name,site_location,timezone_or_utc_offset,latitude,longitude,elevation_m,dc_capacity_kW,kg_climate,pvcz_composite,...,has_power_data,has_current_data,has_voltage_data,has_ac_data,has_dc_data,module_type,simplified_type,system_source,num_days_actual_records,sample_year
2,4,NREL x-Si -1,"Golden, CO",7,39.7406,-105.1774,1795.3,1.0,BSk,12,...,True,True,True,True,True,mono-Si,monocrystalline_Si,PVDAQ General,5063,2008


In [7]:
my_timezones = set(systems_cleaned_enough_ac_data['timezone_or_utc_offset'].unique())
my_timezones

{'5',
 '7',
 'America/Chicago',
 'America/Denver',
 'America/Los_Angeles',
 'America/New_York'}

5 is a synonym for 'Etc/GMT+5', etc.

In [8]:
before_after_dst = {
    'America/Los_Angeles': (8, 7),
    'America/Denver': (7, 6),
    'America/Chicago': (6, 5),
    'America/New_York': (5, 4)
}

# Early attempt

In [9]:
def is_good_timezone_every_year(
    input_dir: str,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    before_after_dst: dict,
    print_messages: bool = True
):
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_ind = relevant_rows_systems.index[0]
    nominal_timezone = systems_cleaned.at[first_ind, 'timezone_or_utc_offset']
    if len(nominal_timezone) == 1:  #single-digit
        nominal_timezone = f'Etc/GMT+{nominal_timezone}'

    
    start_year = 1998
    end_year = 2023
    num_years = end_year - start_year + 1
    # make output range
    data_per_year = pd.DataFrame(
        data = np.full((num_years, 3), pd.NA),
        index = range(start_year, end_year + 1),
        columns= ['good_timezone', 'start_date', 'end_date']
    ) 
    # grab data from sample year
    core_dir = Path(input_dir + f'{system_id}/')  
    for year in range(start_year, end_year + 1):
        current_year_pq = pq.ParquetDataset(
            core_dir,
            filters = [('year', '==', year)]
        )
        current_year_df = current_year_pq.read().to_pandas()
        if current_year_df.shape[0] > 10:
            current_year_df.loc[:, 'date'] = current_year_df['time'].dt.date
            data_per_year.at[year, 'start_date'] = current_year_df['date'].min()
            data_per_year.at[year, 'end_date'] = current_year_df['date'].max()
            # no way of checking if the time too short
            if (
                (data_per_year.at[year, 'end_date'] - data_per_year.at[year, 'start_date'])
                > timedelta(days=30)
            ):
                try:
                    current_year_df['time'] = current_year_df['time'].dt.tz_localize(nominal_timezone)
                    if print_messages:
                        print(f'System {system_id} has a nice timezone of {nominal_timezone} '
                              + f'in the year {year}.')
                    data_per_year.at[year, 'good_timezone'] = True
                except (pytz.NonExistentTimeError, pytz.AmbiguousTimeError):
                    options_tuple = before_after_dst[nominal_timezone]
                    if print_messages:
                        print(f'System {system_id}, year {year}, lies between the options in GMT+{options_tuple}')
                    data_per_year.at[year, 'good_timezone'] = False
                except BaseException as e:
                    raise e
    # drop empty rows
    data_per_year = data_per_year.dropna(axis=0, how='all')
    return data_per_year

In [10]:
systems_with_some_bad_timestamps = []
systems_with_all_good_timestamps = []
for system_id in enough_data_parquet_power_list:
    timezones_per_year = is_good_timezone_every_year(
        '../../../../data_ds_project/testing_yearly_parquet/',
        system_id,
        systems_cleaned,
        before_after_dst,
        False
    )
    bad_timezones_per_year = timezones_per_year[timezones_per_year['good_timezone'] == False]
    if bad_timezones_per_year.shape[0] > 0:
        systems_with_some_bad_timestamps.append(system_id)
        print(f'System {system_id}')
        print('Bad years:')
        print(timezones_per_year)
        print('')
    else:
        systems_with_all_good_timestamps.append(system_id)
        print(f'System {system_id} has good data.')
        print('')

System 4 has good data.

System 10 has good data.

System 33 has good data.

System 34
Bad years:
     good_timezone  start_date    end_date
2010          True  2010-08-23  2010-12-31
2011         False  2011-01-01  2011-12-31
2012         False  2012-01-01  2012-12-31
2013         False  2013-01-01  2013-12-31
2014         False  2014-01-01  2014-12-31
2015         False  2015-01-01  2015-12-31
2016         False  2016-01-01  2016-12-31
2017         False  2017-01-01  2017-12-31
2018         False  2018-01-01  2018-12-31
2019         False  2019-01-01  2019-12-31
2020          True  2020-01-01  2020-07-26

System 35
Bad years:
     good_timezone  start_date    end_date
2010         False  2010-08-23  2010-12-31
2011         False  2011-01-01  2011-12-31
2012         False  2012-01-01  2012-12-31
2013         False  2013-01-01  2013-12-31
2014         False  2014-01-01  2014-12-31
2015         False  2015-01-01  2015-12-31
2016         False  2016-01-01  2016-12-31
2017         False  

Get the dst transition *Dates* (stable for US data)

In [11]:
practice_tz = pytz.timezone('America/New_York')
practice_tz._utc_transition_times

[datetime.datetime(1, 1, 1, 0, 0),
 datetime.datetime(1901, 12, 13, 20, 45, 52),
 datetime.datetime(1918, 3, 31, 7, 0),
 datetime.datetime(1918, 10, 27, 6, 0),
 datetime.datetime(1919, 3, 30, 7, 0),
 datetime.datetime(1919, 10, 26, 6, 0),
 datetime.datetime(1920, 3, 28, 7, 0),
 datetime.datetime(1920, 10, 31, 6, 0),
 datetime.datetime(1921, 4, 24, 7, 0),
 datetime.datetime(1921, 9, 25, 6, 0),
 datetime.datetime(1922, 4, 30, 7, 0),
 datetime.datetime(1922, 9, 24, 6, 0),
 datetime.datetime(1923, 4, 29, 7, 0),
 datetime.datetime(1923, 9, 30, 6, 0),
 datetime.datetime(1924, 4, 27, 7, 0),
 datetime.datetime(1924, 9, 28, 6, 0),
 datetime.datetime(1925, 4, 26, 7, 0),
 datetime.datetime(1925, 9, 27, 6, 0),
 datetime.datetime(1926, 4, 25, 7, 0),
 datetime.datetime(1926, 9, 26, 6, 0),
 datetime.datetime(1927, 4, 24, 7, 0),
 datetime.datetime(1927, 9, 25, 6, 0),
 datetime.datetime(1928, 4, 29, 7, 0),
 datetime.datetime(1928, 9, 30, 6, 0),
 datetime.datetime(1929, 4, 28, 7, 0),
 datetime.datetime(

In [12]:
transfer_days = [
    my_datetime.date() for my_datetime in practice_tz._utc_transition_times
]

Put *relevant* transfers in a DataFrame

In [16]:
start_year = 1994
end_year = 2023
num_years = end_year - start_year + 1
per_year_dates = []
for year in range(start_year, end_year + 1):
    dates_this_year = [
        my_date for my_date in transfer_days
        if ((my_date >= date(year, 1, 1))
            and (my_date) < date(year + 1, 1, 1))
    ]
    dates_this_year = np.array(dates_this_year).reshape((1, 2))
    year_df = pd.DataFrame(
        data = dates_this_year,
        index=[year,],
        columns=['spring_dst_date', 'fall_dst_date']
    )
    per_year_dates.append(year_df)
transfer_days_df = pd.concat(
    per_year_dates
)

In [18]:
transfer_days_df.head()

,spring_dst_date,fall_dst_date
1994,1994-04-03,1994-10-30
1995,1995-04-02,1995-10-29
1996,1996-04-07,1996-10-27
1997,1997-04-06,1997-10-26
1998,1998-04-05,1998-10-25


Add in whether or not there is data on the date in question

In [19]:
def all_timezone_data(
    input_dir: str,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    dst_transition_dates: pd.DataFrame,
    before_after_dst: dict,
    print_messages: bool = True
):
    # make DataFrame
    start_year = 1998
    end_year = 2023
    num_years = end_year - start_year + 1
    # make output range
    data_per_year = pd.DataFrame(
        data = np.full((num_years, 5), pd.NA),
        index = range(start_year, end_year + 1),
        columns= ['good_timezone', 'start_date', 'end_date', 'has_start_dst_data', 'has_end_dst_data']
    ) 
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_ind = relevant_rows_systems.index[0]
    nominal_timezone = systems_cleaned.at[first_ind, 'timezone_or_utc_offset']
    if len(nominal_timezone) == 1:  #no-DST-timestamp.  
        nominal_timezone = f'Etc/GMT+{nominal_timezone}'
        dst_timezone = False
    else:
        dst_timezone = True
    # grab data from sample year
    core_dir = Path(input_dir + f'{system_id}/')  
    for year in range(start_year, end_year + 1):
        current_year_pq = pq.ParquetDataset(
            core_dir,
            filters = [('year', '==', year)]
        )
        current_year_df = current_year_pq.read().to_pandas()
        if current_year_df.shape[0] > 10:
            current_year_df.loc[:, 'date'] = current_year_df['time'].dt.date
            data_per_year.at[year, 'start_date'] = current_year_df['date'].min()
            data_per_year.at[year, 'end_date'] = current_year_df['date'].max()
            spring_transition_date = dst_transition_dates.at[year, 'spring_dst_date']
            fall_transition_date = dst_transition_dates.at[year, 'fall_dst_date']
            spring_data = current_year_df[
                current_year_df['date'] == spring_transition_date
            ]
            data_per_year.at[year, 'has_start_dst_data'] = (spring_data.shape[0] > 0)
            fall_data = current_year_df[
                current_year_df['date'] == fall_transition_date
            ]
            data_per_year.at[year, 'has_end_dst_data'] = (fall_data.shape[0] > 0)
            # if a dst 
            if dst_timezone:
                try:
                    current_year_df['time'] = current_year_df['time'].dt.tz_localize(nominal_timezone)
                    if print_messages:
                        print(f'System {system_id} has a nice timezone of {nominal_timezone} '
                               + f'in the year {year}.')
                    # generally, a good start if has data on DST day, or all data on one side
                    # of the starting transition
                    good_start = (
                        data_per_year.at[year, 'has_start_dst_data']
                        or (data_per_year.at[year, 'start_date'] > spring_transition_date)
                        or (data_per_year.at[year, 'end_date'] < spring_transition_date)
                    )
                    good_end = (
                        data_per_year.at[year, 'has_end_dst_data']
                        or (data_per_year.at[year, 'end_date'] < fall_transition_date)
                        or (data_per_year.at[year, 'start_date'] > fall_transition_date)
                    )
                    # one exception -- if data *only* during between DST days, no information
                    not_middle_only = (
                        (data_per_year.at[year, 'start_date'] < spring_transition_date)
                        or (data_per_year.at[year, 'end_date'] > fall_transition_date)
                    )
                    if good_start and good_end and not_middle_only:
                        data_per_year.at[year, 'good_timezone'] = True
                except (pytz.NonExistentTimeError, pytz.AmbiguousTimeError):
                    options_tuple = before_after_dst[nominal_timezone]
                    if print_messages:
                        print(f'System {system_id}, year {year}, lies between the options in GMT+{options_tuple}')
                    data_per_year.at[year, 'good_timezone'] = False
                except BaseException as e:
                    raise e
            else:  #nominally test, but impossible to fail!
                try:
                    current_year_df['time'] = current_year_df['time'].dt.tz_localize(nominal_timezone)
                    if print_messages:
                        print(f'System {system_id} has a nice timezone of {nominal_timezone} '
                              + f'in the year {year}.')
                    data_per_year.at[year, 'good_timezone'] = True
                except (pytz.NonExistentTimeError, pytz.AmbiguousTimeError):
                    options_tuple = before_after_dst[nominal_timezone]
                    if print_messages:
                        print(f'System {system_id}, year {year}, lies between the options in GMT+{options_tuple}')
                    data_per_year.at[year, 'good_timezone'] = False
                except BaseException as e:
                    raise e
    # drop empty rows and columns
    data_per_year = data_per_year.dropna(axis=0, how='all')
    return data_per_year

In [20]:
for system_id in enough_data_parquet_power_list:
    print(f'System {system_id}')
    print(all_timezone_data(
        '../../../../data_ds_project/testing_yearly_parquet/',
        system_id,
        systems_cleaned,
        transfer_days_df,
        before_after_dst,
        False
    ))
    print('')

System 4
     good_timezone  start_date    end_date has_start_dst_data has_end_dst_data
2007          True  2007-08-26  2007-12-31              False             True
2008          True  2008-01-01  2008-12-31               True             True
2009          True  2009-01-01  2009-06-03               True            False
2010          True  2010-01-22  2010-12-31               True             True
2011          True  2011-01-01  2011-12-31               True             True
2012          True  2012-01-01  2012-12-31               True             True
2013          True  2013-01-01  2013-12-31               True             True
2014          True  2014-01-01  2014-12-31               True             True
2015          True  2015-01-01  2015-12-31               True             True
2016          True  2016-01-01  2016-12-31               True             True
2017          True  2017-01-01  2017-12-31               True             True
2018          True  2018-01-01  2018-12-14 

(Made some slight changes from first version, but checked same output printout!)

In [22]:
some_nulls = []
for system_id in enough_data_parquet_power_list:
    my_data = all_timezone_data(
        '../../../../data_ds_project/testing_yearly_parquet/',
        system_id,
        systems_cleaned,
        transfer_days_df,
        before_after_dst,
        False
    )
    # manually add the weird systems
    my_nulls = my_data[my_data['good_timezone'].isna()]
    if my_nulls.shape[0] > 0:
        print(f'System {system_id}')
        print(my_nulls)
        print('')
        some_nulls.append(system_id)

System 34
     good_timezone  start_date    end_date has_start_dst_data has_end_dst_data
2010          <NA>  2010-08-23  2010-12-31              False            False

System 35
     good_timezone  start_date    end_date has_start_dst_data has_end_dst_data
2020          <NA>  2020-01-14  2020-07-26              False            False

System 1200
     good_timezone  start_date    end_date has_start_dst_data has_end_dst_data
2013          <NA>  2013-01-20  2013-12-31              False             True

System 1277
     good_timezone  start_date    end_date has_start_dst_data has_end_dst_data
2020          <NA>  2020-01-01  2020-07-26              False            False

System 1300
     good_timezone  start_date    end_date has_start_dst_data has_end_dst_data
2013          <NA>  2013-02-14  2013-12-31               True            False
2016          <NA>  2016-01-01  2016-12-31              False             True

System 1307
     good_timezone  start_date    end_date has_start_dst_d

## Towards timezone verification

### Step 1: Sanity check

For systems with no datetime timezone, how much does the starting time (measured) vary over a week?

In [31]:
def time_of_day_in_minutes(time_in: py_time):
    time_h = time_in.hour
    time_m = time_h * 60 + time_in.minute
    time_s = time_m * 60 + time_in.second
    return time_s / 60

In [32]:
time_of_day_in_minutes(py_time(12, 15, 20))

735.3333333333334

In [23]:
def seasons_classifer(
    spring_dst_date: date,
    fall_dst_date: date,
    data_year_start: date,
    data_year_end: date
):
    seasons_dict = {
        'start_only': (data_year_end < spring_dst_date),
        'spring_transfer': (
            (data_year_start < spring_dst_date)
            and (data_year_end > spring_dst_date)
        ),
        'middle_only': (
            (data_year_start > spring_dst_date)
            and (data_year_end < fall_dst_date)
        ),
        'fall_transfer': (
            (data_year_start < fall_dst_date)
            and (data_year_end > fall_dst_date)
        ),
        'end_only': data_year_end > fall_dst_date,
    }
    return seasons_dict

Made-up data to check function

In [24]:
seasons_classifer(
    date(2011, 4, 1),
    date(2011, 10, 1),
    date(2011, 1, 1),
    date(2011, 3, 1)
)

{'start_only': True,
 'spring_transfer': False,
 'middle_only': False,
 'fall_transfer': False,
 'end_only': False}

In [25]:
def data_around_transition(
    input_dir: str,
    system_id: int,
    year: int,
    center_date: date,
    radius: int = 7
):
    # start by going backwards radius days.
    transfer_start = center_date - timedelta(days=radius)
    # convert datetime.date to datetime.datetime
    transfer_start = datetime(
        transfer_start.year,
        transfer_start.month,
        transfer_start.day
    )
    # ditto for upper side
    transfer_end = center_date + timedelta(days=radius)
    transfer_end = datetime(
        transfer_end.year,
        transfer_end.month,
        transfer_end.day
    )
    # load data
    if input_dir[-1] == '/':
        direct_dir = f'{input_dir}{system_id}/'
    else:
        direct_dir = f'{input_dir}/{system_id}/'
    transfer_data_pq = pq.ParquetDataset(
        direct_dir,
        filters = [('year', '==', year),
                   ('time', '>=', transfer_start),
                   ('time', '<=', transfer_end)]
    )
    transfer_data_df = transfer_data_pq.read().to_pandas()
    if transfer_data_df.shape[0] >= 10:
        return transfer_data_df
    else:
        return None

In [35]:
sample_10_spr_2012 = data_around_transition(
    '../../../../data_ds_project/testing_yearly_parquet/',
    10,
    2012,
    transfer_days_df.at[2012, 'spring_dst_date'],
    7
)

In [29]:
sample_1200_spr_2012 = data_around_transition(
    '../../../../data_ds_project/testing_yearly_parquet/',
    1200,
    2012,
    transfer_days_df.at[2012, 'spring_dst_date'],
    7
)

Get starting and ending 'good' data points per day.

In [46]:
def data_range_local(
    df: pd.DataFrame,
):
    power_columns = [
        col_name for col_name in df.columns
        if 'pow' in col_name
    ]
    lower_spread = 0
    upper_spread = 0
    for pow_name in power_columns:
        local_max = df[pow_name].max()
        threshold = 0.01 * local_max
        df_pow = df[df[pow_name] >= threshold].copy(deep=True)
        # errors elsewhere if no good data, so
        if len(df_pow) < 10:
            continue
        df_pow = df_pow[['time', pow_name]]
        df_pow.loc[:, 'date'] = df_pow['time'].dt.date
        df_pow.loc[:, 'time_of_day'] = df_pow['time'].dt.time
        df_pow.loc[:, 'time_of_day_arith'] = df_pow.apply(
            lambda row: time_of_day_in_minutes(row['time_of_day']),
            axis=1
        )
        # grab (minutes of) starting and ending times
        df_pow_daily_terms = df_pow.groupby(['date'])['time_of_day_arith'].agg(['min', 'max'])
        start_min = df_pow_daily_terms['min'].min()
        start_mode = df_pow_daily_terms['min'].value_counts().index[0]
        start_max = df_pow_daily_terms['min'].max()
        loc_lower_spread = max(start_max - start_mode, start_mode - start_min)
        lower_spread = max(lower_spread, loc_lower_spread)
        end_min = df_pow_daily_terms['max'].min()
        end_mode = df_pow_daily_terms['max'].value_counts().index[0]
        end_max = df_pow_daily_terms['max'].max()
        loc_upper_spread = max(end_max - end_mode, end_mode - end_min)
        upper_spread = max(upper_spread, loc_upper_spread)
    return (lower_spread, upper_spread)

In [47]:
data_range_local(sample_10_spr_2012)

(np.float64(91.0), np.float64(106.0))

In [48]:
data_range_local(sample_1200_spr_2012)

(np.float64(215.16666666666663), np.float64(429.9333333333333))

In [54]:
def starting_hour_consistency(
    input_dir: str,
    system_id: int,
    systems_cleaned: pd.DataFrame,
    dst_transition_dates: pd.DataFrame,
    before_after_dst: dict,
    print_messages: bool = True
):
    total_timezone_data = all_timezone_data(
        input_dir,
        system_id,
        systems_cleaned,
        dst_transition_dates,
        before_after_dst,
        False
    )
    relevant_rows_systems = systems_cleaned[systems_cleaned['system_id'] == system_id]
    first_ind = relevant_rows_systems.index[0]
    nominal_timezone = systems_cleaned.at[first_ind, 'timezone_or_utc_offset']
    if len(nominal_timezone) == 1:  #no-DST-timestamp.  
        nominal_timezone = f'Etc/GMT+{nominal_timezone}'
    else:
        # not good for present checking against baseline
        return None
    # make output table
    num_years = len(total_timezone_data.index)
    cols_bool = ['spring_valid', 'fall_valid']
    bool_data = np.full((num_years, 2), fill_value=False)
    bool_df = pd.DataFrame(
        data=bool_data,
        columns=cols_bool,
        index=total_timezone_data.index
    )
    float_data = np.full((num_years, 2), fill_value=np.nan)
    float_cols = ['minutes_spread_up', 'minutes_spread_down']
    float_df = pd.DataFrame(
        data=float_data,
        columns=float_cols,
        index=total_timezone_data.index,
    )
    final_df = pd.merge(left=bool_df, right=float_df,
                          left_index=True, right_index=True)
    for year in total_timezone_data.index:
        spring_transition_date = dst_transition_dates.at[year, 'spring_dst_date']
        fall_transition_date = dst_transition_dates.at[year, 'fall_dst_date']
        start_data = total_timezone_data.at[year, 'start_date']
        end_data = total_timezone_data.at[year, 'end_date']
        # classify data missingness
        seasons_dict = seasons_classifer(
            spring_dst_date=spring_transition_date,
            fall_dst_date=fall_transition_date,
            data_year_start=start_data,
            data_year_end=end_data
        )
        final_df.at[year, 'spring_valid'] = seasons_dict['spring_transfer']
        final_df.at[year, 'fall_valid'] = seasons_dict['fall_transfer']
        spread_down = 0
        spread_up = 0
        if seasons_dict['spring_transfer']:
            # load data for the transition
            spring_data = data_around_transition(
                input_dir,
                system_id,
                year,
                spring_transition_date,
                7
            )
            if spring_data is None:
                if print_messages:
                    print(system_id)
                    print(year)
                    print(spring_transition_date)
                # can happen!  System 33, year 2019, bad data in first half of March!
                final_df.at[year, 'spring_valid'] = False
                # raise RuntimeError(f'Bad transition, system {system_id}, year {year}, spring!')
            else:
                spring_data['time'] = spring_data['time'].dt.tz_localize(nominal_timezone)
                current_spread_down, current_spread_up = data_range_local(spring_data)
                spread_down = max(spread_down, current_spread_down)
                spread_up = max(spread_up, current_spread_up)
        if seasons_dict['fall_transfer']:
            # load data for the transition
            fall_data = data_around_transition(
                input_dir,
                system_id,
                year,
                fall_transition_date,
                7
            )
            if fall_data is None:
                if print_messages:
                    print(system_id)
                    print(year)
                    print(fall_transition_date)
                final_df.at[year, 'fall_valid'] = False
                # raise RuntimeError(f'Bad transition, system {system_id}, year {year}, fall!')
            else:
                fall_data['time'] = fall_data['time'].dt.tz_localize(nominal_timezone)
                current_spread_down, current_spread_up = data_range_local(fall_data)
                spread_down = max(spread_down, current_spread_down)
                spread_up = max(spread_up, current_spread_up)
        final_df.at[year, 'minutes_spread_down'] = spread_down
        final_df.at[year, 'minutes_spread_up'] = spread_up
    return final_df

In [55]:
for system_id in enough_data_parquet_power_list:
    spread_data = starting_hour_consistency(
        '../../../../data_ds_project/testing_yearly_parquet/',
        system_id,
        systems_cleaned,
        transfer_days_df,
        before_after_dst,
        True
    )
    if spread_data is not None:
        print('')
        print(f'System {system_id}')
        print(spread_data)
        print('')


System 4
      spring_valid  fall_valid  minutes_spread_up  minutes_spread_down
2007         False        True               45.0                 60.0
2008          True        True               45.0                105.0
2009          True       False               30.0                 60.0
2010          True        True              224.0                138.0
2011          True        True              155.0                152.0
2012          True        True              112.0                 81.0
2013          True        True              109.0                372.0
2014          True        True              275.0                135.0
2015          True        True               78.0                190.0
2016          True        True               81.0                572.0
2017          True        True             1439.0                  0.0
2018          True        True              252.0                120.0
2019          True        True              266.0                32

In [56]:
# check some validity

In [57]:
system_4_2007_fall = data_around_transition(
    '../../../../data_ds_project/testing_yearly_parquet/',
    4,
    2007,
    transfer_days_df.at[2007, 'fall_dst_date'],
    7
)

In [58]:
system_4_2007_fall.head()

,time,ac_power_kW,year
0,2007-10-28 00:00:00,-0.000158,2007
1,2007-10-28 00:15:00,-0.000158,2007
2,2007-10-28 00:30:00,-0.000158,2007
3,2007-10-28 00:45:00,-0.000158,2007
4,2007-10-28 01:00:00,-0.000158,2007


In [59]:
threshold = system_4_2007_fall['ac_power_kW'].max() * 0.01

In [63]:
system_4_2007_large = system_4_2007_fall[system_4_2007_fall['ac_power_kW'] >= threshold].copy(deep=True)

In [64]:
system_4_2007_large.head()

,time,ac_power_kW,year
28,2007-10-28 07:00:00,0.014169,2007
29,2007-10-28 07:15:00,0.046911,2007
30,2007-10-28 07:30:00,0.059185,2007
31,2007-10-28 07:45:00,0.052128,2007
32,2007-10-28 08:00:00,0.062044,2007


In [70]:
system_4_2007_large.loc[:, 'date'] = system_4_2007_large['time'].dt.date
system_4_2007_large.loc[:, 'time_of_day'] = system_4_2007_large['time'].dt.time

In [72]:
system_4_2007_large.loc[:, 'time_in_min'] = system_4_2007_large.apply(
    lambda row: time_of_day_in_minutes(row['time_of_day']),
    axis=1
)

In [73]:
system_4_2007_large.head()

,time,ac_power_kW,year,date,time_of_day,time_in_min
28,2007-10-28 07:00:00,0.014169,2007,2007-10-28,07:00:00,420.0
29,2007-10-28 07:15:00,0.046911,2007,2007-10-28,07:15:00,435.0
30,2007-10-28 07:30:00,0.059185,2007,2007-10-28,07:30:00,450.0
31,2007-10-28 07:45:00,0.052128,2007,2007-10-28,07:45:00,465.0
32,2007-10-28 08:00:00,0.062044,2007,2007-10-28,08:00:00,480.0


In [76]:
system_4_2007_large_groups = system_4_2007_large.groupby(['date',])['time_in_min']

In [80]:
system_4_2007_start_finish = system_4_2007_large_groups.agg(['min', 'max'])

In [88]:
system_4_2007_start_finish.describe()

,min,max
count,14.000000,14.000000
mean,418.928571,976.071429
std,19.032217,14.958734
min,405.000000,960.000000
25%,405.000000,960.000000
50%,412.500000,975.000000
75%,420.000000,990.000000
max,465.000000,1005.000000


In [81]:
system_4_2007_start_finish['min'].describe()

count     14.000000
mean     418.928571
std       19.032217
min      405.000000
25%      405.000000
50%      412.500000
75%      420.000000
max      465.000000
Name: min, dtype: float64

In [82]:
system_4_2007_start_finish['max'].describe()

count      14.000000
mean      976.071429
std        14.958734
min       960.000000
25%       960.000000
50%       975.000000
75%       990.000000
max      1005.000000
Name: max, dtype: float64

In [83]:
system_4_2007_fall_wide = data_around_transition(
    '../../../../data_ds_project/testing_yearly_parquet/',
    4,
    2007,
    transfer_days_df.at[2007, 'fall_dst_date'],
    14
)
threshold = system_4_2007_fall_wide['ac_power_kW'].max() * 0.01
system_4_2007_fall_wide_large = system_4_2007_fall_wide[system_4_2007_fall_wide['ac_power_kW'] >= threshold].copy(deep=True)
system_4_2007_fall_wide_large.loc[:, 'date'] = system_4_2007_fall_wide_large['time'].dt.date
system_4_2007_fall_wide_large.loc[:, 'time_in_min'] = system_4_2007_fall_wide_large.apply(
    lambda row: time_of_day_in_minutes(row['time']),
    axis=1
)


In [84]:
system_4_2007_larger_groups = system_4_2007_fall_wide_large.groupby(['date',])['time_in_min']

In [85]:
system_4_2007_large_start_finish = system_4_2007_larger_groups.agg(['min', 'max'])

In [86]:
system_4_2007_large_start_finish.head()

,min,max
date,,
2007-10-21,750.0,990.0
2007-10-22,390.0,975.0
2007-10-23,405.0,975.0
2007-10-24,390.0,975.0
2007-10-25,390.0,975.0


In [87]:
system_4_2007_large_start_finish.describe()

,min,max
count,28.000000,28.000000
mean,433.392857,975.535714
std,67.372013,13.217503
min,390.000000,945.000000
25%,405.000000,971.250000
50%,420.000000,975.000000
75%,423.750000,990.000000
max,750.000000,1005.000000
